# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following best practices for referencing Croissant schema entities via their `@id` fields.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

The dataset includes ordered logistic regression outputs for predictors of knowledge adoption in rangeland management in Northern Kenya.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
We first load the dataset metadata and records using `mlcroissant`. This will provide access to all defined RecordSets and their fields for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
We will list all available RecordSets and their fields, referencing entities by their `@id`. These `@id` values are required for subsequent extraction and processing steps.

In [ ]:
# Explore and print available RecordSets and their fields by @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets are defined in the Croissant schema. Please review the schema for record set definitions.")
else:
    for record_set in record_sets:
        print(f"RecordSet: {record_set['@id']} (name: {record_set.get('name', '')})")
        fields = record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"  Field: {field_id}")

## 3. Data Extraction
We will extract records from each record set using its `@id`, and load them into Pandas DataFrames for further analysis. All references use the Croissant `@id` for consistency.

**Note:** If the RecordSet list above is empty, please update the notebook to specify valid RecordSet `@id`s based on the latest schema.

In [ ]:
# Attempt to extract data for each RecordSet. Replace the ids below with valid @id values from your overview.
# For demonstration, this template assumes two possible record set ids (update these as needed):
record_set_ids = [rs['@id'] for rs in getattr(dataset, 'record_sets', [])]
dataframes = {}

if not record_set_ids:
    print("No record sets found for extraction. Please define record set @ids based on your dataset schema.")
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record set: {record_set_id} (num records: {len(dataframes[record_set_id])})")
        except Exception as e:
            print(f"Failed to load records for RecordSet {record_set_id}: {str(e)}")

# Display columns of the first available record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
You can now process, filter, normalize, or group the records. Below, we demonstrate filtering numeric fields, normalizing them, and grouping data by another field.

### Instructions
- Replace `numeric_field_id` and `group_field_id` below with actual field `@id` values from your data overview section.

In [ ]:
# Example EDA: filtering and normalization.
# Update these variable values to match the exact @id of fields in your data.

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Example: List columns for guidance
    print(f"Available columns for {record_set_id}: {df.columns.tolist()}")
    
    # Specify the numeric field @id (update as needed)
    numeric_field_id = None
    for col in df.columns:
        # Attempt to select a numeric-looking column for demonstration
        if df[col].dtype in ['float64', 'int64']:
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric field detected for this RecordSet. Please specify a valid numeric field @id.")
    else:
        # Filtering – keep records with high values for the numeric field
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id]).all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
        print(filtered_df.head())

        # Normalization – z-score
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical column (replace as appropriate)
        possible_groups = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
        group_field_id = possible_groups[0] if possible_groups else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")

## 5. Visualization
Let's visualize the data. We plot the distribution for the selected numeric field, and, if applicable, show mean values grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    # Plot histogram for the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Plot grouped mean if grouping was done
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df.sort_values(numeric_field_id, ascending=False),
                    x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Visualization skipped: No numeric field available or data not loaded.")

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load a Croissant-specified dataset with `mlcroissant`
- Inspect record sets and fields by their `@id`
- Extract data into Pandas DataFrames
- Perform simple EDA, normalization, grouping, and visualization referencing dataset entities by `@id`

For your own analyses, be sure to adapt the field and record set `@id`s to match those specified in your unique dataset schema.